In [ ]:
# ============================================================
# EVALUATION DES DISTANCES 27, 28, 29 ET DU SNB
# ============================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors

from khiops.sklearn import KhiopsClassifier

In [ ]:
# ============================================================
# PARAMETRES
# ============================================================

# Paramètres injectables avec Papermill si nécessaire
DATA_PATH = "Adult.txt"
TARGET_COL = "class"

# Répertoire déjà alimenté par :
# - MakeCI-ponderees-files.ipynb
# - ComputeShapleyValues.ipynb
OUTPUT_DIR = "khiops_output_adult"

SEP = "\t"
OUTPUT_SEP = ";"

ID_COL = None

N_SPLITS = 10
RANDOM_STATE = 42

# Valeurs de k pour le k-PPV
# K_VALUES = [1, 3, 5, 7]
K_VALUES = [1]



# Recalcule le SNB pour disposer des métriques train ET test.
# Ce recalcul utilise les mêmes folds et les mêmes paramètres
# Khiops que MakeCI-ponderees-files.ipynb.
COMPUTE_SNB_METRICS = True

# ------------------------------------------------------------
# COHERENCE PREDICTIVE LOCALE DES VOISINAGES
# ------------------------------------------------------------
#
# Calcule LocDelta_k(d) pour les voisinages test induits par :
# - Eq.27 ;
# - Eq.28 ;
# - Eq.29.
#
# Delta_1(D, D') =
# somme_j |log P(C_j | D) - log P(C_j | D')|.
# ------------------------------------------------------------

COMPUTE_LOCAL_DELTA = True
COMPUTE_LOCAL_AGREE = True

# Protection numérique pour le calcul des logarithmes.
LOG_PROBABILITY_EPS = 1e-15


In [ ]:
# ============================================================
# LECTURE ET PREPARATION DES DONNEES
# ============================================================

DATASET_NAME = Path(DATA_PATH).stem
OUTPUT_DIR = Path(OUTPUT_DIR)

df = pd.read_csv(DATA_PATH, sep=SEP)
df.columns = df.columns.str.strip()

if TARGET_COL not in df.columns:
    raise ValueError(
        f"La colonne cible '{TARGET_COL}' est absente.\n"
        f"Colonnes disponibles : {df.columns.tolist()}"
    )

# Nettoyage homogène des champs texte
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .replace({"?": pd.NA, "": pd.NA})
    )

df = df.dropna(subset=[TARGET_COL]).copy()

if ID_COL is not None:
    if ID_COL not in df.columns:
        raise ValueError(
            f"La colonne ID_COL '{ID_COL}' est absente.\n"
            f"Colonnes disponibles : {df.columns.tolist()}"
        )

    ids = df[ID_COL].copy()
    df_model = df.drop(columns=[ID_COL]).copy()

else:
    # Doit être identique aux deux notebooks précédents.
    ids = pd.Series(df.index, index=df.index, name="row_id")
    df_model = df.copy()

X = df_model.drop(columns=[TARGET_COL]).copy()

y_raw = df_model[TARGET_COL].copy()

if pd.api.types.is_numeric_dtype(y_raw):

    y_numeric = pd.to_numeric(y_raw, errors="raise")

    if (y_numeric.dropna() % 1 == 0).all():
        y = y_numeric.astype("int64")
    else:
        y = y_numeric.astype("float64")

else:

    y = pd.Series(
        y_raw
        .astype("string")
        .str.strip()
        .to_numpy(dtype=str),
        index=y_raw.index,
        name=TARGET_COL,
        dtype=object,
    )

y.name = TARGET_COL

for col in X.columns:
    if (
        pd.api.types.is_string_dtype(X[col])
        or isinstance(X[col].dtype, pd.StringDtype)
    ):
        X[col] = X[col].astype(object)

if y.value_counts().min() < N_SPLITS:
    raise ValueError(
        f"La plus petite classe ne contient que "
        f"{y.value_counts().min()} observations.\n"
        f"Impossible de faire une CV stratifiée à {N_SPLITS} folds."
    )

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print(f"Dataset       : {DATASET_NAME}")
print(f"Observations  : {len(X)}")
print(f"Variables     : {X.shape[1]}")
print(f"Classes       : {sorted(y.unique().tolist())}")
print(f"Nombre de folds : {N_SPLITS}")


In [ ]:
# ============================================================
# FONCTIONS UTILITAIRES
# ============================================================

def read_export(path):
    """Lit un CSV exporté avec le séparateur défini."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable :\n{path}")

    return pd.read_csv(path, sep=OUTPUT_SEP)


def normalize_name(value):
    """
    Supprime espaces, accents indirects, tirets et caractères
    non alphanumériques afin de comparer les noms de colonnes.
    """
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def validate_matrix(df, columns, context):
    """Contrôle et retourne une matrice numérique sans NaN ni infini."""
    if not columns:
        raise ValueError(f"Aucune colonne trouvée pour : {context}")

    missing_columns = [
        column
        for column in columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Colonnes absentes dans {context} :\n"
            f"{missing_columns}"
        )

    matrix = df[columns].apply(pd.to_numeric, errors="raise").copy()

    if matrix.isna().any().any():
        bad_columns = matrix.columns[matrix.isna().any()].tolist()

        raise ValueError(
            f"Valeurs manquantes détectées dans {context}.\n"
            f"Colonnes concernées : {bad_columns}"
        )

    if not np.isfinite(matrix.to_numpy(dtype=float)).all():
        raise ValueError(
            f"Valeurs infinies détectées dans {context}."
        )

    return matrix


def compute_auc_metrics(y_true, probabilities, classes):
    """
    Calcule les métriques AUC.

    Binaire :
      - auc

    Multiclasse :
      - auc_ovr_macro
      - auc_ovr_weighted
    """

    y_true = normalize_target_labels(y_true).to_numpy()
    classes = normalize_target_labels(classes).to_numpy()

    test_classes = np.unique(y_true)

    result = {
        "auc": np.nan,
        "auc_ovr_macro": np.nan,
        "auc_ovr_weighted": np.nan,
        "auc_type": None,
    }

    # L'AUC est indéfinie si le jeu évalué ne contient qu'une classe.
    if len(test_classes) < 2:
        result["auc_type"] = "undefined_one_class"
        return result

    # Cas binaire
    if len(classes) == 2:
        positive_class = classes[1]
        y_binary = (y_true == positive_class).astype(int)
    
        binary_auc = roc_auc_score(
            y_binary,
            probabilities[:, 1],
        )
    
        # Pour deux classes, l'AUC OvR macro est égale à l'AUC binaire.
        result["auc"] = binary_auc
        result["auc_ovr_macro"] = binary_auc
        result["auc_ovr_weighted"] = binary_auc
        result["auc_type"] = "binary"
    
        return result

    # Cas multiclasse : chaque classe doit être présente.
    if set(test_classes) != set(classes):
        result["auc_type"] = "undefined_missing_class"
        return result

    result["auc_ovr_macro"] = roc_auc_score(
        y_true,
        probabilities,
        labels=classes,
        multi_class="ovr",
        average="macro",
    )

    result["auc_ovr_weighted"] = roc_auc_score(
        y_true,
        probabilities,
        labels=classes,
        multi_class="ovr",
        average="weighted",
    )

    result["auc_type"] = "multiclass_ovr"

    return result


def build_class_probabilities(neighbor_labels, classes):
    """
    Construit les probabilités k-PPV à partir des labels des voisins.

    Chaque probabilité est la fréquence d'une classe parmi les k voisins.
    """
    classes = np.asarray(classes, dtype=object)

    n_rows, k = neighbor_labels.shape

    probabilities = np.zeros(
        (n_rows, len(classes)),
        dtype=float,
    )

    for class_index, class_name in enumerate(classes):
        probabilities[:, class_index] = (
            neighbor_labels == class_name
        ).sum(axis=1) / k

    return probabilities


def evaluate_knn_test(
    X_train,
    y_train,
    X_test,
    y_test,
    k_values,
    fold_number,
    representation_name,
):
    """
    k-PPV sur le test :
    - les voisins sont exclusivement recherchés dans le train ;
    - distance Manhattan.
    """
    X_train = np.asarray(X_train, dtype=float)
    X_test = np.asarray(X_test, dtype=float)

    y_train = normalize_target_labels(y_train)
    y_test = normalize_target_labels(y_test)

    classes = np.sort(y_train.unique())
    max_k = max(k_values)

    if len(X_train) < max_k:
        raise ValueError(
            f"Fold {fold_number} : le train contient {len(X_train)} lignes, "
            f"mais k_max vaut {max_k}."
        )

    neighbors = NearestNeighbors(
        n_neighbors=max_k,
        metric="manhattan",
        algorithm="auto",
    )

    neighbors.fit(X_train)

    _, neighbor_indices = neighbors.kneighbors(X_test)

    rows = []

    for k in k_values:
        selected_indices = neighbor_indices[:, :k]

        labels = y_train.iloc[
            selected_indices.reshape(-1)
        ].to_numpy().reshape(len(X_test), k)

        probabilities = build_class_probabilities(labels, classes)
        predictions = classes[np.argmax(probabilities, axis=1)]

        auc_metrics = compute_auc_metrics(
            y_true=y_test,
            probabilities=probabilities,
            classes=classes,
        )

        rows.append(
            {
                "fold": fold_number,
                "split": "test",
                "model": "kNN",
                "representation": representation_name,
                "k": k,
                "n_train_reference": len(X_train),
                "n_evaluated": len(X_test),
                "n_features": X_train.shape[1],
                "accuracy": accuracy_score(y_test, predictions),
                **auc_metrics,
            }
        )

    return pd.DataFrame(rows)


def evaluate_knn_train_loo(
    X_train,
    y_train,
    k_values,
    fold_number,
    representation_name,
):
    """
    k-PPV sur le train avec Leave-One-Out.

    Une observation ne peut pas être son propre voisin.
    Cela évite une accuracy artificiellement égale à 1 pour k=1.
    """
    X_train = np.asarray(X_train, dtype=float)
    y_train = normalize_target_labels(y_train)
    
    n_train = len(X_train)
    max_k = max(k_values)

    if n_train <= max_k:
        raise ValueError(
            f"Fold {fold_number} : il faut au moins k_max + 1 lignes "
            f"pour l'évaluation train Leave-One-Out.\n"
            f"n_train={n_train}, k_max={max_k}"
        )

    classes = np.sort(y_train.unique())

    # On demande k_max + 1 voisins, car l'observation elle-même
    # est présente dans le train et doit être supprimée.
    neighbors = NearestNeighbors(
        n_neighbors=max_k + 1,
        metric="manhattan",
        algorithm="auto",
    )

    neighbors.fit(X_train)

    _, raw_neighbor_indices = neighbors.kneighbors(X_train)

    valid_neighbor_indices = np.zeros(
        (n_train, max_k),
        dtype=int,
    )

    for row_index in range(n_train):
        candidates = raw_neighbor_indices[row_index]

        # Retrait du self-neighbor, identifié par le même index.
        candidates = candidates[candidates != row_index]

        if len(candidates) < max_k:
            raise RuntimeError(
                f"Fold {fold_number}, ligne train {row_index} : "
                f"nombre insuffisant de voisins après retrait de soi-même."
            )

        valid_neighbor_indices[row_index] = candidates[:max_k]

    rows = []

    for k in k_values:
        selected_indices = valid_neighbor_indices[:, :k]

        labels = y_train.iloc[
            selected_indices.reshape(-1)
        ].to_numpy().reshape(n_train, k)

        probabilities = build_class_probabilities(labels, classes)
        predictions = classes[np.argmax(probabilities, axis=1)]

        auc_metrics = compute_auc_metrics(
            y_true=y_train,
            probabilities=probabilities,
            classes=classes,
        )

        rows.append(
            {
                "fold": fold_number,
                "split": "train_loo",
                "model": "kNN",
                "representation": representation_name,
                "k": k,
                "n_train_reference": n_train,
                "n_evaluated": n_train,
                "n_features": X_train.shape[1],
                "accuracy": accuracy_score(y_train, predictions),
                **auc_metrics,
            }
        )

    return pd.DataFrame(rows)






###################################################################
def evaluate_snb(
    classifier,
    X_eval,
    y_eval,
    fold_number,
    split_name,
):
    """Calcule accuracy et AUC du SNB pour un ensemble donné."""
    y_eval = normalize_target_labels(y_eval)

    predictions = normalize_target_labels(
        classifier.predict(X_eval)
    ).to_numpy()
    
    probabilities = classifier.predict_proba(X_eval)
    
    classes = normalize_target_labels(
        classifier.classes_
    ).to_numpy()

    auc_metrics = compute_auc_metrics(
        y_true=y_eval,
        probabilities=probabilities,
        classes=classes,
    )

    return {
        "fold": fold_number,
        "split": split_name,
        "model": "SNB",
        "representation": "native_snb",
        "k": np.nan,
        "n_train_reference": np.nan,
        "n_evaluated": len(y_eval),
        "n_features": X_eval.shape[1],
        "accuracy": accuracy_score(y_eval, predictions),
        **auc_metrics,
    }


################################
def normalize_target_labels(values):
    """
    Normalise les labels de classes pour garantir leur compatibilité
    avec scikit-learn.

    - 0, 1, 2, ... restent numériques ;
    - les modalités textuelles restent des chaînes homogènes.

    Retourne toujours une Series 1D sans type object numérique ambigu.
    """
    labels = pd.Series(values).reset_index(drop=True)

    numeric_labels = pd.to_numeric(labels, errors="coerce")

    # Toutes les modalités sont numériques : Pima, Pendigits,
    # Waveform, Shuttle, etc.
    if numeric_labels.notna().all():

        numeric_array = numeric_labels.to_numpy(dtype=float)

        if np.all(np.equal(numeric_array, np.floor(numeric_array))):
            return pd.Series(
                numeric_array.astype(np.int64),
                name=labels.name,
            )

        return pd.Series(
            numeric_array.astype(np.float64),
            name=labels.name,
        )

    # Au moins une modalité n'est pas numérique :
    # Adult, Vehicle, Mushroom, Letters, Phoneme, etc.
    return pd.Series(
        labels.astype("string")
        .str.strip()
        .to_numpy(dtype=str),
        name=labels.name,
    )


########################################
def evaluate_local_delta_test(
    X_train,
    X_test,
    log_posteriors_train,
    log_posteriors_test,
    k_values,
    fold_number,
    representation_name,
):
    """
    Calcule la cohérence prédictive locale des voisinages induits
    par une représentation.

    Pour chaque observation test D, on recherche ses k plus proches
    voisins D' dans le train selon la distance Manhattan appliquée
    à la représentation considérée.

    Puis on calcule :

        Delta_1(D, D')
        = somme_j |log P(C_j | D) - log P(C_j | D')|

    et :

        LocDelta_k(d)
        = moyenne_D [ moyenne_{D' dans N_k^d(D)} Delta_1(D, D') ]

    Une valeur faible indique que la distance induit des voisinages
    plus cohérents du point de vue des probabilités prédites par SNB.
    """

    X_train = np.asarray(X_train, dtype=float)
    X_test = np.asarray(X_test, dtype=float)

    log_posteriors_train = np.asarray(
        log_posteriors_train,
        dtype=float,
    )

    log_posteriors_test = np.asarray(
        log_posteriors_test,
        dtype=float,
    )

    if X_train.shape[0] != log_posteriors_train.shape[0]:
        raise ValueError(
            "Incohérence entre le nombre de lignes train de la "
            "représentation et les probabilités a posteriori."
        )

    if X_test.shape[0] != log_posteriors_test.shape[0]:
        raise ValueError(
            "Incohérence entre le nombre de lignes test de la "
            "représentation et les probabilités a posteriori."
        )

    if log_posteriors_train.shape[1] != log_posteriors_test.shape[1]:
        raise ValueError(
            "Le nombre de classes diffère entre train et test."
        )

    max_k = max(k_values)

    if len(X_train) < max_k:
        raise ValueError(
            f"Fold {fold_number} : n_train={len(X_train)} est "
            f"inférieur à k_max={max_k}."
        )

    # Les voisins sont déterminés exclusivement dans le train.
    neighbors = NearestNeighbors(
        n_neighbors=max_k,
        metric="manhattan",
        algorithm="auto",
    )

    neighbors.fit(X_train)

    _, neighbor_indices = neighbors.kneighbors(X_test)

    # Pour chaque observation test et chacun de ses voisins train :
    #
    # Delta_1(D, D') =
    # sum_j | log P(C_j | D) - log P(C_j | D') |
    #
    # Forme obtenue :
    # (n_test, max_k, n_classes)
    delta_components = np.abs(
        log_posteriors_test[:, np.newaxis, :]
        - log_posteriors_train[neighbor_indices, :]
    )

    # Forme :
    # (n_test, max_k)
    delta_by_neighbor = delta_components.sum(axis=2)

    rows = []

    for k in k_values:

        # Moyenne des Delta_1 sur les k voisins de chaque test.
        local_delta_by_test_observation = (
            delta_by_neighbor[:, :k]
            .mean(axis=1)
        )

        rows.append(
            {
                "fold": fold_number,
                "split": "test",
                "model": "kNN_neighborhood",
                "representation": representation_name,
                "k": k,
                "n_train_reference": len(X_train),
                "n_evaluated": len(X_test),
                "n_features": X_train.shape[1],
                "locdelta": float(
                    local_delta_by_test_observation.mean()
                ),
                "locdelta_std_test_observations": float(
                    local_delta_by_test_observation.std(ddof=0)
                ),
            }
        )

    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# VERIFICATION DES FICHIERS NECESSAIRES
# ============================================================

fold_number = 1
fold_dir = OUTPUT_DIR / f"fold_{fold_number:02d}"

required_paths = [
    (
        fold_dir
        / f"{DATASET_NAME}_train_fold_{fold_number:02d}_conditional_info_weighted.csv"
    ),
    (
        fold_dir
        / f"{DATASET_NAME}_test_fold_{fold_number:02d}_conditional_info_weighted.csv"
    ),
    (
        fold_dir
        / f"{DATASET_NAME}_train_fold_{fold_number:02d}_shapley.csv"
    ),
    (
        fold_dir
        / f"{DATASET_NAME}_test_fold_{fold_number:02d}_shapley.csv"
    ),
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"Le fichier requis est absent :\n{path}\n\n"
            "Exécutez d'abord les notebooks :\n"
            " - MakeCI-ponderees-files.ipynb\n"
            " - ComputeShapleyValues.ipynb"
        )

print("Tous les fichiers requis du fold 1 sont disponibles.")


In [ ]:
# ============================================================
# CALCULS PAR FOLD
# ============================================================

all_knn_results = []

all_snb_results = []

# Stockage des valeurs LocDelta_k par fold et par distance.
all_local_delta_results = []


for fold_number, (train_index, test_index) in enumerate(
    cv.split(X, y),
    start=1,
):

    print("\n" + "=" * 80)
    print(f"FOLD {fold_number}/{N_SPLITS}")
    print("=" * 80)

    fold_dir = OUTPUT_DIR / f"fold_{fold_number:02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Reconstruction des jeux bruts du fold.
    #
    # Ils sont nécessaires pour :
    # - l'évaluation du SNB ;
    # - predict_proba, utilisé pour le calcul de Delta_1.
    # --------------------------------------------------------

    X_train_raw = X.iloc[train_index].copy()
    X_test_raw = X.iloc[test_index].copy()

    y_train_raw = y.iloc[train_index].copy()
    y_test_raw = y.iloc[test_index].copy()

    # --------------------------------------------------------
    # Chargement des Conditional Info pondérées.
    # --------------------------------------------------------

    ci_train_path = (
        fold_dir
        / (
            f"{DATASET_NAME}_train_fold_{fold_number:02d}_"
            "conditional_info_weighted.csv"
        )
    )

    ci_test_path = (
        fold_dir
        / (
            f"{DATASET_NAME}_test_fold_{fold_number:02d}_"
            "conditional_info_weighted.csv"
        )
    )

    ci_train = read_export(ci_train_path)
    ci_test = read_export(ci_test_path)

    # --------------------------------------------------------
    # Chargement des valeurs de Shapley.
    # --------------------------------------------------------

    shapley_train_path = (
        fold_dir
        / f"{DATASET_NAME}_train_fold_{fold_number:02d}_shapley.csv"
    )

    shapley_test_path = (
        fold_dir
        / f"{DATASET_NAME}_test_fold_{fold_number:02d}_shapley.csv"
    )

    shapley_train = read_export(shapley_train_path)
    shapley_test = read_export(shapley_test_path)

    # --------------------------------------------------------
    # Contrôles de cohérence des exports.
    # --------------------------------------------------------

    required_base_columns = {
        "id",
        "fold",
        TARGET_COL,
    }

    for dataframe_name, dataframe in {
        "ci_train": ci_train,
        "ci_test": ci_test,
        "shapley_train": shapley_train,
        "shapley_test": shapley_test,
    }.items():

        missing_columns = (
            required_base_columns
            - set(dataframe.columns)
        )

        if missing_columns:
            raise ValueError(
                f"Colonnes absentes dans {dataframe_name} : "
                f"{sorted(missing_columns)}"
            )

        found_folds = dataframe["fold"].unique()

        if (
            len(found_folds) != 1
            or found_folds[0] != fold_number
        ):
            raise ValueError(
                f"Numéro de fold incohérent dans "
                f"{dataframe_name} : {found_folds}. "
                f"Fold attendu : {fold_number}"
            )

    # Les exports CI et Shapley doivent être dans le même ordre.
    if not ci_train["id"].equals(shapley_train["id"]):
        raise ValueError(
            f"Les ids CI et Shapley train diffèrent "
            f"au fold {fold_number}."
        )

    if not ci_test["id"].equals(shapley_test["id"]):
        raise ValueError(
            f"Les ids CI et Shapley test diffèrent "
            f"au fold {fold_number}."
        )

    if not ci_train[TARGET_COL].equals(shapley_train[TARGET_COL]):
        raise ValueError(
            f"Les classes CI et Shapley train diffèrent "
            f"au fold {fold_number}."
        )

    if not ci_test[TARGET_COL].equals(shapley_test[TARGET_COL]):
        raise ValueError(
            f"Les classes CI et Shapley test diffèrent "
            f"au fold {fold_number}."
        )

    # Les labels de référence sont lus depuis les exports afin
    # d'assurer une compatibilité exacte avec les représentations.
    y_train = normalize_target_labels(ci_train[TARGET_COL])
    y_test = normalize_target_labels(ci_test[TARGET_COL])

    classes = np.sort(y_train.unique())

    print(f"Classes : {classes.tolist()}")
    print(f"Train   : {len(y_train)} lignes")
    print(f"Test    : {len(y_test)} lignes")

    # ========================================================
    # EQUATION 27
    #
    # d_global_1(D, D')
    #
    # =
    #
    # d_NB_1(D, D')
    #
    # + J * |log p_w(D) - log p_w(D')|
    #
    # Distance Manhattan sur :
    #
    # - les Conditional Info pondérées :
    #     w_m * log p(D_m | C_j)
    #
    # - la coordonnée :
    #     J * log p_w(D).
    # ========================================================

    weighted_ci_columns = [
        column
        for column in ci_train.columns
        if column.startswith("weighted_")
    ]

    if not weighted_ci_columns:
        raise ValueError(
            f"Aucune colonne 'weighted_' trouvée "
            f"au fold {fold_number}."
        )

    global_term_column = "global_log_p_D_weighted_x_J"

    if global_term_column not in ci_train.columns:
        raise ValueError(
            f"Colonne absente du train : {global_term_column}\n"
            "Relancez MakeCI-ponderees-files.ipynb."
        )

    if global_term_column not in ci_test.columns:
        raise ValueError(
            f"Colonne absente du test : {global_term_column}\n"
            "Relancez MakeCI-ponderees-files.ipynb."
        )

    eq27_columns = (
        weighted_ci_columns
        + [global_term_column]
    )

    missing_eq27_columns = [
        column
        for column in eq27_columns
        if column not in ci_test.columns
    ]

    if missing_eq27_columns:
        raise ValueError(
            f"Colonnes Eq.27 absentes du test, "
            f"fold {fold_number} :\n"
            f"{missing_eq27_columns}"
        )

    X_train_eq27 = validate_matrix(
        ci_train,
        eq27_columns,
        context=f"equation 27 train fold {fold_number}",
    )

    X_test_eq27 = validate_matrix(
        ci_test,
        eq27_columns,
        context=f"equation 27 test fold {fold_number}",
    )

    print(
        f"Equation 27 : "
        f"{X_train_eq27.shape[1]} composantes"
    )

    all_knn_results.append(
        evaluate_knn_train_loo(
            X_train=X_train_eq27,
            y_train=y_train,
            k_values=K_VALUES,
            fold_number=fold_number,
            representation_name=(
                "equation_27_weighted_conditional_info"
            ),
        )
    )

    all_knn_results.append(
        evaluate_knn_test(
            X_train=X_train_eq27,
            y_train=y_train,
            X_test=X_test_eq27,
            y_test=y_test,
            k_values=K_VALUES,
            fold_number=fold_number,
            representation_name=(
                "equation_27_weighted_conditional_info"
            ),
        )
    )

    # ========================================================
    # EQUATION 28
    #
    # d_odds_1(D, D')
    #
    # =
    #
    # d_Shap_1(D, D')
    #
    # Distance Manhattan sur le vecteur complet des valeurs de
    # Shapley, sans agrégation préalable entre les classes.
    # ========================================================

    eq28_columns = [
        column
        for column in shapley_train.columns
        if column.startswith("shapley_")
    ]

    if not eq28_columns:
        raise ValueError(
            f"Aucune colonne Shapley trouvée au fold "
            f"{fold_number}."
        )

    missing_eq28_columns = [
        column
        for column in eq28_columns
        if column not in shapley_test.columns
    ]

    if missing_eq28_columns:
        raise ValueError(
            f"Colonnes Eq.28 absentes du test, "
            f"fold {fold_number} :\n"
            f"{missing_eq28_columns}"
        )

    X_train_eq28 = validate_matrix(
        shapley_train,
        eq28_columns,
        context=f"equation 28 train fold {fold_number}",
    )

    X_test_eq28 = validate_matrix(
        shapley_test,
        eq28_columns,
        context=f"equation 28 test fold {fold_number}",
    )

    print(
        f"Equation 28 : "
        f"{X_train_eq28.shape[1]} composantes Shapley"
    )

    all_knn_results.append(
        evaluate_knn_train_loo(
            X_train=X_train_eq28,
            y_train=y_train,
            k_values=K_VALUES,
            fold_number=fold_number,
            representation_name=(
                "equation_28_shapley_log_odds"
            ),
        )
    )

    all_knn_results.append(
        evaluate_knn_test(
            X_train=X_train_eq28,
            y_train=y_train,
            X_test=X_test_eq28,
            y_test=y_test,
            k_values=K_VALUES,
            fold_number=fold_number,
            representation_name=(
                "equation_28_shapley_log_odds"
            ),
        )
    )

    # ========================================================
    # EQUATION 29 = EQUATION 27 SANS LE TERME GLOBAL
    #
    # d_1^Eq29(D, D')
    #
    # =
    #
    # somme_m somme_j
    #
    # w_m * |log P(D_m | C_j) - log P(D'_m | C_j)|
    #
    # Eq.29 est donc strictement identique à la distance
    # pondérée de l'équation (6).
    #
    # Eq.27 : Conditional Info pondérées
    #         + J * log p_w(D)
    #
    # Eq.29 : Conditional Info pondérées seules.
    # ========================================================

    eq29_columns = weighted_ci_columns.copy()

    missing_eq29_columns = [
        column
        for column in eq29_columns
        if column not in ci_test.columns
    ]

    if missing_eq29_columns:
        raise ValueError(
            f"Colonnes Eq.29 absentes du test, "
            f"fold {fold_number} :\n"
            f"{missing_eq29_columns}"
        )

    X_train_eq29 = validate_matrix(
        ci_train,
        eq29_columns,
        context=f"equation 29 / equation 6 train fold {fold_number}",
    )

    X_test_eq29 = validate_matrix(
        ci_test,
        eq29_columns,
        context=f"equation 29 / equation 6 test fold {fold_number}",
    )

    print(
        "Equation 29 (= Eq.6, sans terme global) : "
        f"{X_train_eq29.shape[1]} composantes"
    )

    all_knn_results.append(
        evaluate_knn_train_loo(
            X_train=X_train_eq29,
            y_train=y_train,
            k_values=K_VALUES,
            fold_number=fold_number,
            representation_name=(
                "equation_29_weighted_conditional_info_no_global"
            ),
        )
    )

    all_knn_results.append(
        evaluate_knn_test(
            X_train=X_train_eq29,
            y_train=y_train,
            X_test=X_test_eq29,
            y_test=y_test,
            k_values=K_VALUES,
            fold_number=fold_number,
            representation_name=(
                "equation_29_weighted_conditional_info_no_global"
            ),
        )
    )

    # ========================================================
    # SNB KHIOPS
    #
    # Le SNB est appris uniquement sur les données train du fold.
    #
    # Il sert :
    # - aux métriques prédictives du SNB ;
    # - aux probabilités a posteriori nécessaires pour Delta_1.
    # ========================================================

    if COMPUTE_SNB_METRICS or COMPUTE_LOCAL_DELTA:

        print(
            "Apprentissage du SNB pour les métriques / "
            "LocDelta..."
        )

        snb_eval_dir = (
            fold_dir
            / "snb_metrics_evaluation_classifier"
        )

        snb_classifier = KhiopsClassifier(
            n_trees=0,
            n_pairs=0,
            output_dir=str(snb_eval_dir),
        )

        snb_classifier.fit(
            X_train_raw,
            y_train_raw,
        )

        # ----------------------------------------------------
        # COHERENCE PREDICTIVE LOCALE DES VOISINAGES
        #
        # LocDelta_k(Eq.27), LocDelta_k(Eq.28), LocDelta_k(Eq.29)
        # ----------------------------------------------------

        if COMPUTE_LOCAL_DELTA:

            posterior_probabilities_train = (
                snb_classifier.predict_proba(X_train_raw)
            )

            posterior_probabilities_test = (
                snb_classifier.predict_proba(X_test_raw)
            )

            # Delta_1 est calculée sur les log-probabilités
            # a posteriori. Le clipping évite log(0).
            log_posteriors_train = np.log(
                np.clip(
                    posterior_probabilities_train,
                    LOG_PROBABILITY_EPS,
                    1.0,
                )
            )

            log_posteriors_test = np.log(
                np.clip(
                    posterior_probabilities_test,
                    LOG_PROBABILITY_EPS,
                    1.0,
                )
            )

            # Eq.27
            all_local_delta_results.append(
                evaluate_local_delta_test(
                    X_train=X_train_eq27,
                    X_test=X_test_eq27,
                    log_posteriors_train=log_posteriors_train,
                    log_posteriors_test=log_posteriors_test,
                    k_values=K_VALUES,
                    fold_number=fold_number,
                    representation_name=(
                        "equation_27_weighted_conditional_info"
                    ),
                )
            )

            # Eq.28 : valeurs de Shapley / log-odds.
            all_local_delta_results.append(
                evaluate_local_delta_test(
                    X_train=X_train_eq28,
                    X_test=X_test_eq28,
                    log_posteriors_train=log_posteriors_train,
                    log_posteriors_test=log_posteriors_test,
                    k_values=K_VALUES,
                    fold_number=fold_number,
                    representation_name=(
                        "equation_28_shapley_log_odds"
                    ),
                )
            )

            # Eq.29 : Eq.27 sans le terme global, donc Eq.6.
            all_local_delta_results.append(
                evaluate_local_delta_test(
                    X_train=X_train_eq29,
                    X_test=X_test_eq29,
                    log_posteriors_train=log_posteriors_train,
                    log_posteriors_test=log_posteriors_test,
                    k_values=K_VALUES,
                    fold_number=fold_number,
                    representation_name=(
                        "equation_29_weighted_conditional_info_no_global"
                    ),
                )
            )

        # ----------------------------------------------------
        # METRIQUES PREDICTIVES DU SNB NATIF
        # ----------------------------------------------------

        if COMPUTE_SNB_METRICS:

            all_snb_results.append(
                evaluate_snb(
                    classifier=snb_classifier,
                    X_eval=X_train_raw,
                    y_eval=y_train_raw,
                    fold_number=fold_number,
                    split_name="train",
                )
            )

            all_snb_results.append(
                evaluate_snb(
                    classifier=snb_classifier,
                    X_eval=X_test_raw,
                    y_eval=y_test_raw,
                    fold_number=fold_number,
                    split_name="test",
                )
            )

    # ========================================================
    # EXPORTS DES RESULTATS DU FOLD
    # ========================================================

    fold_knn_results = pd.concat(
        [
            result
            for result in all_knn_results
            if result["fold"].iloc[0] == fold_number
        ],
        ignore_index=True,
    )

    fold_knn_path = (
        fold_dir
        / (
            f"{DATASET_NAME}_knn_metrics_train_test_"
            f"fold_{fold_number:02d}.csv"
        )
    )

    fold_knn_results.to_csv(
        fold_knn_path,
        index=False,
        sep=OUTPUT_SEP,
    )

    if COMPUTE_SNB_METRICS:

        fold_snb_results = pd.DataFrame(
            [
                result
                for result in all_snb_results
                if result["fold"] == fold_number
            ]
        )

        fold_snb_path = (
            fold_dir
            / (
                f"{DATASET_NAME}_snb_metrics_train_test_"
                f"fold_{fold_number:02d}.csv"
            )
        )

        fold_snb_results.to_csv(
            fold_snb_path,
            index=False,
            sep=OUTPUT_SEP,
        )

    if COMPUTE_LOCAL_DELTA:

        fold_local_delta_results = pd.concat(
            [
                result
                for result in all_local_delta_results
                if result["fold"].iloc[0] == fold_number
            ],
            ignore_index=True,
        )

        fold_local_delta_path = (
            fold_dir
            / (
                f"{DATASET_NAME}_local_delta_test_"
                f"fold_{fold_number:02d}.csv"
            )
        )

        fold_local_delta_results.to_csv(
            fold_local_delta_path,
            index=False,
            sep=OUTPUT_SEP,
        )

    print("\nFichiers créés pour ce fold :")
    print(f" - {fold_knn_path}")

    if COMPUTE_SNB_METRICS:
        print(f" - {fold_snb_path}")

    if COMPUTE_LOCAL_DELTA:
        print(f" - {fold_local_delta_path}")


In [ ]:
# ============================================================
# EXPORTS GLOBAUX
# ============================================================

# ------------------------------------------------------------
# Résultats k-PPV : Eq.27, Eq.28 et Eq.29
# ------------------------------------------------------------

knn_results_all_folds = pd.concat(
    all_knn_results,
    ignore_index=True,
)

knn_results_all_folds = (
    knn_results_all_folds
    .sort_values(
        by=[
            "representation",
            "split",
            "k",
            "fold",
        ]
    )
    .reset_index(drop=True)
)

knn_all_folds_path = (
    OUTPUT_DIR
    / f"{DATASET_NAME}_knn_metrics_train_test_all_folds.csv"
)

knn_results_all_folds.to_csv(
    knn_all_folds_path,
    index=False,
    sep=OUTPUT_SEP,
)


# ------------------------------------------------------------
# Résultats du SNB natif
# ------------------------------------------------------------

if COMPUTE_SNB_METRICS:

    snb_results_all_folds = pd.DataFrame(
        all_snb_results
    )

    snb_results_all_folds = (
        snb_results_all_folds
        .sort_values(
            by=[
                "split",
                "fold",
            ]
        )
        .reset_index(drop=True)
    )

    snb_all_folds_path = (
        OUTPUT_DIR
        / f"{DATASET_NAME}_snb_metrics_train_test_all_folds.csv"
    )

    snb_results_all_folds.to_csv(
        snb_all_folds_path,
        index=False,
        sep=OUTPUT_SEP,
    )


# ------------------------------------------------------------
# Résultats LocDelta :
# cohérence prédictive locale des voisinages test.
# ------------------------------------------------------------

if COMPUTE_LOCAL_DELTA:

    local_delta_results_all_folds = pd.concat(
        all_local_delta_results,
        ignore_index=True,
    )

    local_delta_results_all_folds = (
        local_delta_results_all_folds
        .sort_values(
            by=[
                "representation",
                "k",
                "fold",
            ]
        )
        .reset_index(drop=True)
    )

    # 3 distances × nombre de k × nombre de folds.
    expected_rows = (
        3
        * len(K_VALUES)
        * N_SPLITS
    )

    if len(local_delta_results_all_folds) != expected_rows:
        raise RuntimeError(
            "Nombre de lignes LocDelta incohérent.\n\n"
            f" - lignes obtenues : {len(local_delta_results_all_folds)}\n"
            f" - lignes attendues : {expected_rows}\n\n"
            "Pour chaque fold, il faut une ligne par "
            "représentation Eq.27 / Eq.28 / Eq.29 "
            "et par valeur de k."
        )

    local_delta_all_folds_path = (
        OUTPUT_DIR
        / f"{DATASET_NAME}_local_delta_test_all_folds.csv"
    )

    local_delta_results_all_folds.to_csv(
        local_delta_all_folds_path,
        index=False,
        sep=OUTPUT_SEP,
    )


# ------------------------------------------------------------
# Affichage des fichiers créés
# ------------------------------------------------------------

print("=" * 80)
print("EXPORTS GLOBAUX CREES")
print("=" * 80)

print(f" - {knn_all_folds_path}")

if COMPUTE_SNB_METRICS:
    print(f" - {snb_all_folds_path}")

if COMPUTE_LOCAL_DELTA:
    print(f" - {local_delta_all_folds_path}")


In [ ]:
# ============================================================
# RECAPITULATIF DES FOLDS : K-PPV
# ============================================================

print("=" * 80)
print("RESULTATS K-PPV PAR FOLD")
print("=" * 80)

display(knn_results_all_folds)

if COMPUTE_LOCAL_DELTA:

    print("\n" + "=" * 80)
    print("RESULTATS LocDelta PAR FOLD")
    print("=" * 80)

    display(local_delta_results_all_folds)




In [ ]:
# ============================================================
# SYNTHESE K-PPV : MOYENNE ET ECART-TYPE
# ============================================================

knn_summary = (
    knn_results_all_folds
    .groupby(
        ["model", "representation", "split", "k"],
        dropna=False,
    )
    .agg(
        n_folds=("fold", "nunique"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        auc_ovr_macro_mean=("auc_ovr_macro", "mean"),
        auc_ovr_macro_std=("auc_ovr_macro", "std"),
        auc_ovr_weighted_mean=("auc_ovr_weighted", "mean"),
        auc_ovr_weighted_std=("auc_ovr_weighted", "std"),
    )
    .reset_index()
)

knn_summary_path = (
    OUTPUT_DIR
    / f"{DATASET_NAME}_knn_summary_mean_std_train_test.csv"
)

knn_summary.to_csv(
    knn_summary_path,
    index=False,
    sep=OUTPUT_SEP,
)

print("=" * 80)
print("SYNTHESE K-PPV : MOYENNE ± ECART-TYPE SUR LES FOLDS")
print("=" * 80)

display(knn_summary)

print(f"\nFichier créé :\n - {knn_summary_path}")


In [ ]:
# ============================================================
# RESULTATS ET SYNTHESE SNB
# ============================================================

if COMPUTE_SNB_METRICS:

    print("=" * 80)
    print("RESULTATS SNB PAR FOLD")
    print("=" * 80)

    display(snb_results_all_folds)

    snb_summary = (
        snb_results_all_folds
        .groupby(
            ["model", "representation", "split"],
            dropna=False,
        )
        .agg(
            n_folds=("fold", "nunique"),
            accuracy_mean=("accuracy", "mean"),
            accuracy_std=("accuracy", "std"),
            auc_mean=("auc", "mean"),
            auc_std=("auc", "std"),
            auc_ovr_macro_mean=("auc_ovr_macro", "mean"),
            auc_ovr_macro_std=("auc_ovr_macro", "std"),
            auc_ovr_weighted_mean=("auc_ovr_weighted", "mean"),
            auc_ovr_weighted_std=("auc_ovr_weighted", "std"),
        )
        .reset_index()
    )

    snb_summary_path = (
        OUTPUT_DIR
        / f"{DATASET_NAME}_snb_summary_mean_std_train_test.csv"
    )

    snb_summary.to_csv(
        snb_summary_path,
        index=False,
        sep=OUTPUT_SEP,
    )

    print("=" * 80)
    print("SYNTHESE SNB : MOYENNE ± ECART-TYPE SUR LES FOLDS")
    print("=" * 80)

    display(snb_summary)

    print(f"\nFichier créé :\n - {snb_summary_path}")


In [ ]:
# ============================================================
# COMPARAISON FINALE SUR LE TEST
# ============================================================

comparison_test = knn_summary[
    knn_summary["split"] == "test"
].copy()

if COMPUTE_SNB_METRICS:

    snb_test_comparison = snb_summary[
        snb_summary["split"] == "test"
    ].copy()

    comparison_test = pd.concat(
        [comparison_test, snb_test_comparison],
        ignore_index=True,
        sort=False,
    )

comparison_test = comparison_test.sort_values(
    by=["model", "representation", "k"],
    na_position="last",
).reset_index(drop=True)

print("=" * 80)
print("COMPARAISON FINALE SUR LES FOLDS TEST")
print("=" * 80)

display(comparison_test)

comparison_test_path = (
    OUTPUT_DIR
    / f"{DATASET_NAME}_comparison_test_mean_std.csv"
)

comparison_test.to_csv(
    comparison_test_path,
    index=False,
    sep=OUTPUT_SEP,
)

print(f"\nFichier créé :\n - {comparison_test_path}")


In [ ]:
# ============================================================
# COMPARAISON FINALE SUR LE TRAIN
# ============================================================

# Résultats k-PPV sur le train évalué en Leave-One-Out
comparison_train = knn_summary[
    knn_summary["split"] == "train_loo"
].copy()

# Pour rendre le tableau plus lisible :
# train_loo devient train_loo_knn.
comparison_train["split"] = "train_loo_knn"

if COMPUTE_SNB_METRICS:

    # Résultats SNB sur son jeu d'apprentissage
    snb_train_comparison = snb_summary[
        snb_summary["split"] == "train"
    ].copy()

    comparison_train = pd.concat(
        [
            comparison_train,
            snb_train_comparison,
        ],
        ignore_index=True,
        sort=False,
    )

comparison_train = comparison_train.sort_values(
    by=["model", "representation", "k"],
    na_position="last",
).reset_index(drop=True)

print("=" * 80)
print("COMPARAISON FINALE SUR LES FOLDS TRAIN")
print("=" * 80)

print(
    "\nNote méthodologique :\n"
    " - k-NN / train_loo_knn : évaluation Leave-One-Out sur le train ;\n"
    "   chaque observation est exclue de ses propres voisins.\n"
    " - SNB / train : prédiction du SNB sur son jeu d'apprentissage.\n"
)

display(comparison_train)

comparison_train_path = (
    OUTPUT_DIR
    / f"{DATASET_NAME}_comparison_train_mean_std.csv"
)

comparison_train.to_csv(
    comparison_train_path,
    index=False,
    sep=OUTPUT_SEP,
)

print(f"\nFichier créé :\n - {comparison_train_path}")
